# Cognex In-Sight: download the current image over FTP

First working sample used by the team to read images from the Cognex In-Sight camera in the lab.

**What it does**
1. Asks for the camera IP address and credentials (nothing is stored in the notebook).
2. Connects to the camera's FTP server (port 21) and downloads `image.bmp`, the image currently held by the camera.
3. Decodes it with OpenCV (grayscale), saves a timestamped copy in `cognex_captures/` and displays it.

**Notes**
- This downloads the *existing* image; it does not trigger a new capture.
- FTP sends credentials in plain text: use it only on the isolated lab network.
- Never commit the IP address, username or password to the repository.
- A reusable command-line version is available in `vision/cognex/cognex_ftp_capture.py`.


In [ ]:
from ftplib import FTP
from io import BytesIO
from pathlib import Path
from datetime import datetime
from getpass import getpass

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Enter the camera's actual IP address shown in In-Sight Explorer
CAMERA_IP = input("Cognex IP address: ").strip()

# Enter the camera's authorized login credentials
USERNAME = input("Camera username: ").strip()
PASSWORD = getpass("Camera password: ")

# Download the current active image via FTP
# This downloads the existing image; it does not trigger a new capture
image_buffer = BytesIO()

with FTP() as ftp:
    ftp.connect(CAMERA_IP, port=21, timeout=10)
    ftp.login(user=USERNAME, passwd=PASSWORD)
    ftp.retrbinary("RETR image.bmp", image_buffer.write)

# Convert the downloaded bytes into an OpenCV image
image_data = image_buffer.getvalue()
image = cv2.imdecode(
    np.frombuffer(image_data, dtype=np.uint8),
    cv2.IMREAD_GRAYSCALE
)

if image is None:
    raise RuntimeError("The camera did not return a valid BMP image.")

# Save the downloaded file
output_folder = Path.cwd() / "cognex_captures"
output_folder.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
file_path = output_folder / f"cognex_{timestamp}.bmp"
file_path.write_bytes(image_data)

# Display the result
print("Image received. Height and width:", image.shape)
print("Saved to:", file_path.resolve())

plt.figure(figsize=(9, 6))
plt.imshow(image, cmap="gray", vmin=0, vmax=255)
plt.title("Image received from the Cognex camera")
plt.axis("off")
plt.show()